In [1]:
from __future__ import annotations
import time
import numpy as np
from dataclasses import dataclass,field
from typing import Literal,Optional

In [2]:
from dataclasses import dataclass,field

@dataclass
class Config:
  vocab_size: int = 32000
  hidden_dim: int =512
  n_heads: int = 8
  n_layers: int = 4
  ffn_dim: int = 2048
  max_seq_len : int = 128
  dtype: Literal["fp32", "fp16", "int8"] = "fp16"
  attention_kernel: Literal["gemm", "flash"] = "flash"
  max_new_tokens: int = 20
  seed: int = 0

In [3]:
def head_dim(self) -> int:
        assert self.hidden_dim % self.n_heads == 0, "hidden_dim must divide n_heads"
        return self.hidden_dim // self.n_heads


In [4]:

class Profiler:
    """Tracks wall-clock time spent in each named stage."""

    def __init__(self):
        self.records: dict[str, float] = {}
        self.counts: dict[str, int] = {}

    class _Span:
        def __init__(self, profiler: "Profiler", name: str):
            self.profiler = profiler
            self.name = name

        def __enter__(self):
            self.t0 = time.perf_counter()
            return self

        def __exit__(self, *exc):
            dt = time.perf_counter() - self.t0
            self.profiler.records[self.name] = self.profiler.records.get(self.name, 0.0) + dt
            self.profiler.counts[self.name] = self.profiler.counts.get(self.name, 0) + 1

    def span(self, name: str) -> "Profiler._Span":
        return Profiler._Span(self, name)

    def report(self) -> str:
        total = sum(self.records.values()) or 1e-9
        lines = ["=" * 52, "PERFORMANCE REPORT", "=" * 52]
        lines.append(f"{'Stage':<22}{'Calls':>8}{'Time(ms)':>12}{'Share':>10}")
        lines.append("-" * 52)
        for name, t in sorted(self.records.items(), key=lambda kv: -kv[1]):
            calls = self.counts[name]
            lines.append(f"{name:<22}{calls:>8}{t*1000:>12.3f}{t/total*100:>9.1f}%")
        lines.append("-" * 52)
        lines.append(f"{'TOTAL':<22}{'':>8}{total*1000:>12.3f}{100.0:>9.1f}%")
        lines.append("=" * 52)
        return "\n".join(lines)

In [5]:
import math

class MemoryManager:
    """Tracks simulated allocations (weights, activations, KV cache)."""

    DTYPE_BYTES = {"fp32": 4, "fp16": 2, "int8": 1}

    def __init__(self, dtype: str):
        self.dtype = dtype
        self.bytes_per_elem = self.DTYPE_BYTES[dtype]
        self.allocations: dict[str, int] = {}
        self.kv_cache_elems = 0

    def allocate(self, name: str, *shape: int):
        n_elems = math.prod(shape)
        self.allocations[name] = n_elems * self.bytes_per_elem
        return n_elems

    def grow_kv_cache(self, n_layers: int, n_heads: int, head_dim: int, new_tokens: int):

        self.kv_cache_elems += 2 * n_layers * n_heads * head_dim * new_tokens
        self.allocations["kv_cache"] = self.kv_cache_elems * self.bytes_per_elem

    def total_bytes(self) -> int:
        return sum(self.allocations.values())

    def report(self) -> str:
        lines = ["-" * 40, "MEMORY MANAGER", "-" * 40]
        for name, b in self.allocations.items():
            lines.append(f"{name:<20}{b/1e6:>10.3f} MB")
        lines.append("-" * 40)
        lines.append(f"{'TOTAL':<20}{self.total_bytes()/1e6:>10.3f} MB")
        return "\n".join(lines)

In [6]:
class Quantizer:
    """Simulates precision-dependent numeric behavior (not real bit-packing)."""

    def __init__(self, dtype: str):
        self.dtype = dtype

    def apply(self, x: np.ndarray) -> np.ndarray:
        if self.dtype == "fp32":
            return x.astype(np.float32)
        if self.dtype == "fp16":
            return x.astype(np.float16).astype(np.float32)
        if self.dtype == "int8":
            scale = np.max(np.abs(x)) / 127.0 + 1e-8
            q = np.round(x / scale).astype(np.int8)
            return (q.astype(np.float32)) * scale
        raise ValueError(f"Unknown dtype {self.dtype}")


In [7]:
class ModelWeights:
    def __init__(self, cfg: Config, mem: MemoryManager, rng: np.random.Generator):
        H, F, L, V = cfg.hidden_dim, cfg.ffn_dim, cfg.n_layers, cfg.vocab_size

        def rand(*shape):
            return (rng.standard_normal(shape) * 0.02).astype(np.float32)

        self.embedding = rand(V, H)
        mem.allocate("embedding", V, H)

        self.layers = []
        for i in range(L):
            layer = {
                "wq": rand(H, H), "wk": rand(H, H), "wv": rand(H, H), "wo": rand(H, H),
                "w1": rand(H, F), "w2": rand(F, H),
                "ln1_g": np.ones(H, dtype=np.float32), "ln1_b": np.zeros(H, dtype=np.float32),
                "ln2_g": np.ones(H, dtype=np.float32), "ln2_b": np.zeros(H, dtype=np.float32),
            }
            self.layers.append(layer)
            mem.allocate(f"layer{i}_attn", H * H * 4)
            mem.allocate(f"layer{i}_ffn", H * F + F * H)

        self.out_proj = rand(H, V)
        mem.allocate("output_projection", H, V)


In [8]:
class ModelLoader:
    def __init__(self, cfg: Config, mem: MemoryManager, profiler: Profiler):
        self.cfg = cfg
        self.mem = mem
        self.profiler = profiler

    def load(self) -> ModelWeights:
        with self.profiler.span("model_loading"):
            rng = np.random.default_rng(self.cfg.seed)
            weights = ModelWeights(self.cfg, self.mem, rng)
        return weights

In [9]:
def layer_norm(x: np.ndarray, gamma: np.ndarray, beta: np.ndarray, eps: float = 1e-5) -> np.ndarray:
    mu = x.mean(axis=-1, keepdims=True)
    var = x.var(axis=-1, keepdims=True)
    return (x - mu) / np.sqrt(var + eps) * gamma + beta


def softmax(x: np.ndarray) -> np.ndarray:
    x = x - x.max(axis=-1, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=-1, keepdims=True)


In [10]:
class AttentionEngine:
    """Implements attention with two selectable kernel paths."""

    def __init__(self, cfg: Config, quant: Quantizer, profiler: Profiler, mem: MemoryManager):
        self.cfg = cfg
        self.quant = quant
        self.profiler = profiler
        self.mem = mem

    def _split_heads(self, x: np.ndarray) -> np.ndarray:
        T, H = x.shape
        nh, hd = self.cfg.n_heads, self.cfg.head_dim
        return x.reshape(T, nh, hd).transpose(1, 0, 2)

    def _merge_heads(self, x: np.ndarray) -> np.ndarray:
        nh, T, hd = x.shape
        return x.transpose(1, 0, 2).reshape(T, nh * hd)

    def cuda_gemm_attention(self, q, k, v) -> np.ndarray:
        """Naive O(T^2) attention -- simulates a plain GEMM-based kernel."""
        with self.profiler.span("cuda_gemm"):
            scale = 1.0 / math.sqrt(self.cfg.head_dim)
            scores = np.einsum("hqd,hkd->hqk", q, k) * scale
            probs = softmax(scores)
            out = np.einsum("hqk,hkd->hqd", probs, v)
        return out

    def flash_attention(self, q, k, v, block: int = 32) -> np.ndarray:
        """Simulated FlashAttention: fused, block-wise, streaming softmax.
        Functionally equivalent output to the naive kernel, but processes
        keys/values in tiles to emulate reduced memory traffic."""
        with self.profiler.span("flash_attention"):
            nh, T, hd = q.shape
            scale = 1.0 / math.sqrt(hd)
            out = np.zeros_like(q)
            row_max = np.full((nh, T), -np.inf, dtype=np.float32)
            row_sum = np.zeros((nh, T), dtype=np.float32)

            for start in range(0, T, block):
                end = min(start + block, T)
                k_blk, v_blk = k[:, start:end], v[:, start:end]
                scores = np.einsum("hqd,hkd->hqk", q, k_blk) * scale

                blk_max = scores.max(axis=-1)
                new_max = np.maximum(row_max, blk_max)

                exp_scores = np.exp(scores - new_max[..., None])
                correction = np.exp(row_max - new_max)

                out = out * correction[..., None] + np.einsum("hqk,hkd->hqd", exp_scores, v_blk)
                row_sum = row_sum * correction + exp_scores.sum(axis=-1)
                row_max = new_max

            out = out / row_sum[..., None]
        return out

    def forward(self, x: np.ndarray, layer_w: dict, cache: Optional[dict] = None) -> np.ndarray:
        T, H = x.shape
        q = self.quant.apply(x @ layer_w["wq"])
        k = self.quant.apply(x @ layer_w["wk"])
        v = self.quant.apply(x @ layer_w["wv"])

        q, k, v = self._split_heads(q), self._split_heads(k), self._split_heads(v)

        if cache is not None:
            k = np.concatenate([cache["k"], k], axis=1) if cache.get("k") is not None else k
            v = np.concatenate([cache["v"], v], axis=1) if cache.get("v") is not None else v
            cache["k"], cache["v"] = k, v
            self.mem.grow_kv_cache(1, self.cfg.n_heads, self.cfg.head_dim, T)

        if self.cfg.attention_kernel == "flash":
            attn_out = self.flash_attention(q, k, v)
        else:
            attn_out = self.cuda_gemm_attention(q, k, v)

        merged = self._merge_heads(attn_out)
        with self.profiler.span("layernorm"):
            proj = merged @ layer_w["wo"]
        return proj


In [11]:
class FFNEngine:
    def __init__(self, quant: Quantizer, profiler: Profiler):
        self.quant = quant
        self.profiler = profiler

    def forward(self, x: np.ndarray, layer_w: dict) -> np.ndarray:
        with self.profiler.span("ffn"):
            h = self.quant.apply(x @ layer_w["w1"])
            h = np.maximum(h, 0)  # ReLU
            out = h @ layer_w["w2"]
        return out


In [12]:
class TransformerEngine:
    def __init__(self, cfg: Config, weights: ModelWeights, quant: Quantizer,
                 profiler: Profiler, mem: MemoryManager):
        self.cfg = cfg
        self.weights = weights
        self.profiler = profiler
        self.attn = AttentionEngine(cfg, quant, profiler, mem)
        self.ffn = FFNEngine(quant, profiler)
        self.kv_caches = [dict() for _ in range(cfg.n_layers)]

    def embed(self, token_ids: np.ndarray) -> np.ndarray:
        with self.profiler.span("embedding"):
            x = self.weights.embedding[token_ids]
        return x

    def forward(self, token_ids: np.ndarray) -> np.ndarray:
        x = self.embed(token_ids)
        for i, layer_w in enumerate(self.weights.layers):
            normed = layer_norm(x, layer_w["ln1_g"], layer_w["ln1_b"])
            attn_out = self.attn.forward(normed, layer_w, cache=self.kv_caches[i])
            x = x + attn_out
            normed2 = layer_norm(x, layer_w["ln2_g"], layer_w["ln2_b"])
            ffn_out = self.ffn.forward(normed2, layer_w)
            x = x + ffn_out
        return x



In [13]:

class TokenGenerator:
    def __init__(self, cfg: Config, weights: ModelWeights, engine: TransformerEngine,
                 profiler: Profiler, rng: np.random.Generator):
        self.cfg = cfg
        self.weights = weights
        self.engine = engine
        self.profiler = profiler
        self.rng = rng

    def output_projection(self, hidden_last: np.ndarray) -> np.ndarray:
        with self.profiler.span("output_projection"):
            logits = hidden_last @ self.weights.out_proj
        return logits

    def generate(self, prompt_ids: list[int]) -> list[int]:
        tokens = list(prompt_ids)
        current_ids = np.array(tokens, dtype=np.int64)


        hidden = self.engine.forward(current_ids)
        logits = self.output_projection(hidden[-1])
        next_id = int(np.argmax(logits))
        tokens.append(next_id)

        for _ in range(self.cfg.max_new_tokens - 1):
            hidden = self.engine.forward(np.array([next_id], dtype=np.int64))
            logits = self.output_projection(hidden[-1])
            next_id = int(np.argmax(logits))
            tokens.append(next_id)
            if len(tokens) >= self.cfg.max_seq_len:
                break
        return tokens


In [14]:

class RuntimeScheduler:
    def __init__(self, cfg: Config):
        self.cfg = cfg
        self.profiler = Profiler()
        self.mem = MemoryManager(cfg.dtype)
        self.quant = Quantizer(cfg.dtype)

    def run(self, prompt_ids: list[int]) -> dict:
        loader = ModelLoader(self.cfg, self.mem, self.profiler)
        weights = loader.load()

        engine = TransformerEngine(self.cfg, weights, self.quant, self.profiler, self.mem)
        rng = np.random.default_rng(self.cfg.seed)
        generator = TokenGenerator(self.cfg, weights, engine, self.profiler, rng)

        with self.profiler.span("total_generation"):
            output_ids = generator.generate(prompt_ids)

        return {
            "input_ids": prompt_ids,
            "output_ids": output_ids,
            "generated_ids": output_ids[len(prompt_ids):],
        }



In [15]:
def main():
    cfg = Config(
        vocab_size=32000,
        hidden_dim=256,
        n_heads=8,
        n_layers=4,
        ffn_dim=1024,
        max_seq_len=64,
        dtype="fp16",
        attention_kernel="flash",
        max_new_tokens=16,
        seed=42,
    )

    assert cfg.hidden_dim % cfg.n_heads == 0, "hidden_dim must divide n_heads"
    cfg.head_dim = cfg.hidden_dim // cfg.n_heads

    print("Config:", cfg)
    scheduler = RuntimeScheduler(cfg)


    rng = np.random.default_rng(1)
    prompt_ids = rng.integers(0, cfg.vocab_size, size=8).tolist()
    print("\nInput token ids:", prompt_ids)

    result = scheduler.run(prompt_ids)

    print("\nGenerated token ids:", result["generated_ids"])
    print("\n" + scheduler.mem.report())
    print("\n" + scheduler.profiler.report())
    return result

if __name__ == "__main__":
    main()

Config: Config(vocab_size=32000, hidden_dim=256, n_heads=8, n_layers=4, ffn_dim=1024, max_seq_len=64, dtype='fp16', attention_kernel='flash', max_new_tokens=16, seed=42)

Input token ids: [15142, 16378, 24165, 30414, 1115, 4613, 26334, 30356]

Generated token ids: [18194, 18194, 18194, 18194, 18194, 18194, 18194, 18194, 18194, 18194, 18194, 18194, 18194, 18194, 18194, 18194]

----------------------------------------
MEMORY MANAGER
----------------------------------------
embedding               16.384 MB
layer0_attn              0.524 MB
layer0_ffn               1.049 MB
layer1_attn              0.524 MB
layer1_ffn               1.049 MB
layer2_attn              0.524 MB
layer2_ffn               1.049 MB
layer3_attn              0.524 MB
layer3_ffn               1.049 MB
output_projection       16.384 MB
kv_cache                 0.094 MB
----------------------------------------
TOTAL                   39.154 MB

PERFORMANCE REPORT
Stage                    Calls    Time(ms)     Share
--